# Chennai BA/DA Market Intelligence

## Portfolio case study
This notebook analyses the **synthetic/illustrative 120-row dataset** in `sample_listings_chennai.csv`. It demonstrates a Business Analyst / Data Analyst workflow: data validation, KPI definition, feature engineering, descriptive analysis and decision-oriented interpretation.

> **Important:** This is not live scraped labour-market data and should not be presented as a factual survey of Chennai hiring.

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv('sample_listings_chennai.csv')
df['Salary Midpoint LPA'] = (df['Min Salary LPA'] + df['Max Salary LPA']) / 2
df['Salary Spread LPA'] = df['Max Salary LPA'] - df['Min Salary LPA']
df.head()


## 1. Data quality checks


In [ ]:
print('Rows:', len(df))
print('Columns:', len(df.columns))
print('Duplicate rows:', df.duplicated().sum())
print('\nMissing values:')
display(df.isna().sum().to_frame('missing'))
print('Unique companies:', df['Company'].nunique())
print('Unique locations:', df['Location'].nunique())


## 2. Executive KPIs


In [ ]:
skill_series = df['Skills'].str.split(';').explode().str.strip()
kpis = {
    'Sampled listings': len(df),
    'Median salary midpoint (LPA)': round(df['Salary Midpoint LPA'].median(), 2),
    'Top role': df['Job Title'].value_counts().idxmax(),
    'Top skill': skill_series.value_counts().idxmax(),
}
pd.Series(kpis)


## 3. Salary analysis
Salary midpoint is used as a transparent portfolio metric: `(Min Salary + Max Salary) / 2`.


In [ ]:
salary_by_role = (df.groupby('Job Title')
    .agg(Listings=('Job Title','size'),
         Median_Midpoint_LPA=('Salary Midpoint LPA','median'),
         Avg_Min_LPA=('Min Salary LPA','mean'),
         Avg_Max_LPA=('Max Salary LPA','mean'))
    .sort_values('Median_Midpoint_LPA', ascending=False))
display(salary_by_role.round(2))


## 4. Skill demand
A skill is counted at most once per listing, then divided by the number of listings.


In [ ]:
skill_demand = skill_series.value_counts().rename('Listings').to_frame()
skill_demand['Pct_of_listings'] = (skill_demand['Listings'] / len(df) * 100).round(1)
display(skill_demand.head(15))


## 5. Hiring hubs, work arrangement, industry and role mix


In [ ]:
for col in ['Location','Work Arrangement','Industry','Job Title']:
    out = df[col].value_counts().rename_axis(col).reset_index(name='Listings')
    out['Pct'] = (out['Listings']/len(df)*100).round(1)
    print(f'\n{col}')
    display(out)


## 6. BA vs DA skill profile
The chart uses the share of listings within each role that mention a skill. This is a demand percentage, not a skill score.


In [ ]:
ba_da = df[df['Job Title'].isin(['Business Analyst','Data Analyst'])].copy()
long = ba_da.assign(Skill=ba_da['Skills'].str.split(';')).explode('Skill')
long['Skill'] = long['Skill'].str.strip()
role_sizes = ba_da['Job Title'].value_counts()
profile = long.groupby(['Job Title','Skill']).size().rename('Listings').reset_index()
profile['DemandPct'] = (profile['Listings'] / profile['Job Title'].map(role_sizes) * 100).round(1)
display(profile.pivot(index='Skill', columns='Job Title', values='DemandPct').fillna(0).sort_index())


## 7. Decision-oriented interpretation
Use the generated tables to answer: which roles dominate the sample, which skills recur most often, which locations and industries are most represented, and how Business Analyst requirements differ from Data Analyst requirements. Because the source is synthetic, frame all conclusions as **illustrative signals from this sample** rather than factual Chennai-market claims.
